# Session 4 — seed replicate of the 3B run (`v14_seed1`)

`v14_qwen3b` came back as the only arm that escaped the hallucination /
over-refusal tradeoff. It is also a single seed — and this project measured an
18.5 pp seed spread at 1.5B, wide enough to manufacture exactly that kind of
finding. So the good news gets held to the same bar as the bad news.

This session trains **one** run and generates **one** arm. Everything else is
pre-seeded in the dataset so it is skipped.

**~3.5h total.** Setup → GPU T4 x2, Internet **on**, Add Data → the
`refusal-calibration` dataset. Run the pip cell (one expected restart), then
Run All. Scoring happens on a laptop afterwards from `results.zip`.

In [ ]:
!pip install -q unsloth trl peft datasets transformers accelerate pyyaml

Bootstrap: copy the uploaded repo into the working dir and onto the path. **Run this before the rest.**

In [ ]:
import shutil, os, sys, glob
dst = '/kaggle/working/repo'
if os.path.exists(os.path.join(dst, 'tests.py')):
    print('reusing existing', dst)          # mid-session rerun: keep trained work
else:
    if os.path.exists(dst):
        shutil.rmtree(dst)
    hits = glob.glob('/kaggle/input/**/stages.py', recursive=True)
    assert hits, 'stages.py not found under /kaggle/input — add your dataset as Input'
    src = os.path.dirname(hits[0])
    print('copying repo from', src)
    shutil.copytree(src, dst)
os.chdir(dst)
for p in (dst, os.path.join(dst, 'data')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('cwd:', os.getcwd(), '| has tests.py:', os.path.exists('tests.py'))
print('pre-seeded arms (these get skipped):', sorted(os.path.basename(os.path.dirname(p))
      for p in glob.glob('runs/*/responses.jsonl')))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'tests.py'], check=True)  # logic intact first

In [ ]:
from stages import Ctx, train, generate
ctx = Ctx()
assert ctx.ready, 'no frozen eval in the dataset — data/eval.jsonl is required'

In [ ]:
# ~2h40m on a T4: 3B, 1298 steps at ~7.3 s/step. Resumes nothing — if it dies,
# rerun the notebook and it starts this run over (no adapter on disk yet).
trained, broken = train(ctx, ['v14_seed1'])
print('trained:', trained, '| broken:', broken)

In [ ]:
# Only v14_seed1 lacks a responses.jsonl, so this generates exactly one arm
# (~30 min on 3B) and skips base/prompt/base_3b/prompt_3b.
generate(ctx)

In [ ]:
import glob, os
from runner import stage
print('failed stages:', stage.failed or 'none')
print('arms with responses:', sorted(os.path.basename(os.path.dirname(p))
      for p in glob.glob('runs/*/responses.jsonl')))
partial = [os.path.basename(os.path.dirname(p)) for p in glob.glob('runs/*/responses.partial')]
print('unfinished (resumable):', partial or 'none')

## Save — the adapter is kept this time, so the run never has to be repeated

In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/results.zip     runs/v14_seed1/responses.jsonl runs/v14_seed1/meta.json 2>/dev/null
!cd /kaggle/working/repo && zip -qr /kaggle/working/adapter_v14_seed1.zip     runs/v14_seed1/adapter 2>/dev/null
!ls -lh /kaggle/working/*.zip